#### Students Name and ID:
- Daniel Branco, 20220599
- Fernando Cruz, 20220646

## Main Methodology Idea
For our project, we thought about developing two dataframes for each test assignment group, test and control. Both dataframes had the same features only differing in the label, one label was named control_label the other test_label.
We applied both this dataframes on a randomforest, then came up with a way of comparing them and selecting the best features for our problem and apply these features in a clustering algorithm in order to obtain customer segments. We then applied these segments to code given us by the professors to check whether or not they were successfull in identifying a good group of customers to send vouchers to.

In [0]:
pip install mlflow

Python interpreter will be restarted.
  Created wheel for databricks-cli: filename=databricks_cli-0.17.7-py3-none-any.whl size=143878 sha256=97df54fe2080670edf16f94c9fe554dabc9a1d7a5f9a898610d41815bdf754b2
  Stored in directory: /root/.cache/pip/wheels/b6/90/68/94d223a35a3910c1512a8d42d9f8333ce567ef26e250a56227
Successfully built databricks-cli
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.1.1
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e94123b2-cdbd-4d3a-974a-241e9a398cd1
    Can't uninstall 'typing-extensions'. No files were found to uninstall.
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 2.0.1
    Not uninstalling markupsafe at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e94123b2-cdbd-4d3a-974a-241e9a398cd1
    Can't uninstall

In [0]:
import mlflow

In [0]:
!python3 -m pip freeze --disable-pip-version-check --exclude-editable --no-cache-dir > requirements.txt

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.dataframe import DataFrame

In [0]:
orders_df = spark.read.parquet(f"/FileStore/bdm_data_train/train/ord_sample")
ab_test_df = spark.read.parquet(f"/FileStore/bdm_data_train/train/mkt_test_assignment_sample")
sessions_df = spark.read.parquet(f"/FileStore/bdm_data_train/train/sess_sample")
items_df = spark.read.parquet(f"/FileStore/bdm_data_train/train/items_sample")

#### Step 1: Generating features and labels

In this step we generated the dataframes used for model training, we created features and labels and performed a bit of data treatment.

In [0]:
def generate_features(input_df: DataFrame, start_date: str, end_date: str) -> DataFrame:
    """
    Generate features from the input DataFrame.

    :param input_df: Input DataFrame containing order information.
    :param start_date: Start date for filtering the input DataFrame.
    :param end_date: End date for filtering the input DataFrame.

    :return: DataFrame with generated features.
    """
    if all(col in input_df.columns for col in ["order_id", "order_content"]):
        features_df = (
            input_df.withColumn("quantity", f.explode("order_content.quantity"))
            .groupBy("order_id", "order_content")
            .agg(
                f.sum("quantity").alias("total_items_in_order"),
                f.size(f.expr("filter(order_content, x -> x.quantity is not null)")).alias("total_unique_items_in_order")
            )
            .drop("order_content")
        )
        input_df = input_df.join(features_df, ["order_id"], "left")
    elif all(col in input_df.columns for col in ["account_id", "order_id", "city", "distance_merchant_customer", "order_total", "delivery_fee", "discount_vouchers_used"]):
        features_df = (
            input_df.groupBy("account_id")
            .agg(
                f.countDistinct("order_id").alias("order_frequency"),
                f.expr("mode(city)").alias("usual_city"),
                f.avg("delivery_fee").alias("average_delivery_fee"),
                f.avg("distance_merchant_customer").alias("average_distance_merchant_customer"),
                f.avg("order_total").alias("average_order_total"),
                f.sum("discount_vouchers_used").alias("total_discount_vouchers_used"),
                f.sum("subsidy").alias("total_subsidy")
            )
        )
        input_df = input_df.join(features_df, ["account_id"], "left")
    elif all(col in input_df.columns for col in ["account_id", "started_at", "closed_at"]):
        features_df = (
            input_df.groupBy("account_id")
            .agg(
                f.count("account_id").alias("total_sessions"),
                f.sum(f.expr("unix_timestamp(closed_at) - unix_timestamp(started_at)")).alias("total_session_duration")
            )
        )
        input_df = input_df.join(features_df, ["account_id"], "left")
    else:
        raise ValueError("Invalid input DataFrame")

    return input_df

In [0]:
def generate_labels(input_df: DataFrame, data_type: str) -> DataFrame:

    if data_type == 'test':
        
        input_df = input_df.withColumn('test_label',
                                     f.when((f.col('order_id').isNull()), 0)
                                     .when((f.col('order_id').isNotNull()), 1)
                                     .otherwise(None)
                                    )
        labels_df = input_df.groupBy('account_id').agg(f.expr('mode(test_label)').alias('test_label'))
    elif data_type == 'control':
        input_df = input_df.withColumn('control_label',
                                           f.when((f.col('order_id').isNull()), 0)
                                           .when((f.col('order_id').isNotNull()), 1)
                                           .otherwise(None)
                                          )
        labels_df = input_df.groupBy('account_id').agg(f.expr('mode(control_label)').alias('control_label'))
    return labels_df

In [0]:
orders_all = orders_df.filter((f.col('reference_date') >= '2021-01-01') & (f.col('reference_date') < '2021-05-31'))
sessions_all = sessions_df.filter((f.col('reference_date') >= '2021-01-01') & (f.col('reference_date') < '2021-05-31'))
items_all = items_df.filter((f.col('reference_date') >= '2021-01-01') & (f.col('reference_date') < '2021-05-31'))

In [0]:
sessions_all = sessions_all.dropna(subset=['closed_at'])

In [0]:
items_all = generate_features(items_all, '2021-01-01', '2021-05-31')
orders_all = generate_features(orders_all, '2021-01-01', '2021-05-31')
sessions_all = generate_features(sessions_all, '2021-01-01', '2021-05-31')

In [0]:
# Assuming df1, df2, and df3 are the three DataFrames

# Selecting specific columns from df1
orders_select = orders_all.select("account_id", "order_id", "reference_date", "order_frequency", "usual_city", "average_delivery_fee", "average_order_total", "average_distance_merchant_customer", "total_discount_vouchers_used", "total_subsidy")

# Selecting specific columns from df2
items_select = items_all.select("order_id", "total_items_in_order", "total_unique_items_in_order")

# Selecting specific columns from df3
sessions_select = sessions_all.select("account_id", "reference_date","total_sessions", "total_session_duration")

# Joining the selected DataFrames
final_df = orders_select.join(items_select, on="order_id", how="inner").join(sessions_select, on=['account_id','reference_date'], how="left")


In [0]:
def generate_item_averages(input_df: DataFrame, start_date: str, end_date: str) -> DataFrame:
    """
    Generate features from the input DataFrame.

    :param input_df: Input DataFrame containing order information.
    :param start_date: Start date for filtering the input DataFrame.
    :param end_date: End date for filtering the input DataFrame.

    :return: DataFrame with generated features.
    """
    if all(col in input_df.columns for col in ["account_id", "total_items_in_order","total_unique_items_in_order"]):
        features_df = (
            input_df
            .groupBy("account_id")
            .agg(
                f.avg("total_items_in_order").alias("average_total_items_in_order"),
                f.avg("total_unique_items_in_order").alias("average_total_unique_items_in_order"),
            )
            .drop("order_content")
        )
        input_df = input_df.join(features_df, ["account_id"], "left")
    

    return input_df

In [0]:
final_df = generate_item_averages(final_df, '2021-01-01', '2021-05-31')

In [0]:
# Assuming df1, df2, and df3 are the three DataFrames

# Selecting specific columns from df1
final_df = final_df.select("account_id", "order_id", "reference_date", "order_frequency", "usual_city", "average_delivery_fee", "average_order_total", "average_distance_merchant_customer", "total_discount_vouchers_used", "total_subsidy","average_total_items_in_order","average_total_unique_items_in_order","total_sessions", "total_session_duration")


In [0]:
final_df = final_df.dropna(subset=['total_sessions'])

In [0]:
final_df = final_df.dropDuplicates()

In [0]:
orders_june = orders_df.filter((f.col('reference_date') >= '2021-06-01') & (f.col('reference_date') < '2021-06-31'))

In [0]:
test_orders = ab_test_df.join(orders_june, on=['account_id','reference_date'], how='left')

#ver agora e contar para cada user cada vez que esteve no controlo e nao comprou, controlo e comprou, teste e comprou e teste e nao comprou

In [0]:
# Assuming you have a DataFrame 'df' with a column 'type' containing 'test' or 'control' values
test_df = test_orders.filter(test_orders['test_assignment'] == 'test')
control_df = test_orders.filter(test_orders['test_assignment'] == 'control')


In [0]:
test_df = generate_labels(test_df, "test")
control_df = generate_labels(control_df, "control")

In [0]:
final_test = final_df.join(test_df, on="account_id", how="inner")
final_control = final_df.join(control_df, on="account_id", how="inner")

In [0]:
final_test = final_test.select("account_id", "order_frequency", "usual_city", "average_delivery_fee", "average_order_total", "average_distance_merchant_customer", "total_discount_vouchers_used", "average_total_items_in_order", "average_total_unique_items_in_order", "total_sessions", "total_session_duration","total_subsidy","test_label")

In [0]:
final_control = final_control.select("account_id", "order_frequency", "usual_city", "average_delivery_fee", "average_order_total", "average_distance_merchant_customer", "total_discount_vouchers_used", "average_total_items_in_order", "average_total_unique_items_in_order", "total_sessions", "total_session_duration","total_subsidy","control_label")

In [0]:
final_test = final_test.dropDuplicates(["account_id"])

In [0]:
final_control = final_control.dropDuplicates(["account_id"])

In [0]:
output_path = "dbfs:/FileStore/bdm_data_train/test"


In [0]:
dbutils.fs.mkdirs(output_path)

Out[26]: True

In [0]:
final_test.write.mode("overwrite").parquet(output_path + "/final_test.parquet")


In [0]:
output_path = "dbfs:/FileStore/bdm_data_train/control"

In [0]:
dbutils.fs.mkdirs(output_path)

Out[29]: True

In [0]:
final_control.write.mode("overwrite").parquet(output_path + "/final_control.parquet")

> **Features documentation**
- control_label: Label for the control group of the test assignment, 0 for people who did not buy, 1 for those who bought
- test_label: Label for the test group of the test assignment, 0 for people who did not buy, 1 for those who bought
- order_frequency: Count the number of orders for each account_id.
- usual_city: Most frequent city where orders came from.
- average_delivery_fee: Calculate the average delivery_fee for each account_id.
- average_order_total: Calculate the average order_total for each account_id.
- average_distance_merchant_customer: Calculate the average distance_merchant_customer for each account_id.
- total_discount_vouchers_used: Sum the discount_vouchers_used column for each account_id.
- average_total_items_in_order: Calculate the average of the items in each order in each account_id.
- average_total_unique_items_in_order: Calculate the average of unique items in each order in each account_id.
- total_sessions: Count the number of sessions for each account_id.
- total_session_duration: Calculate the sum of the duration (closed_at - started_at) for each account_id.
- total_subsidy: Sum the subsidy column for each account_id.

#### Step 2: The ML Pipeline

####Test Pipeline

In [0]:
final_test = spark.read.parquet("dbfs:/FileStore/bdm_data_train/test/final_test.parquet")

In [0]:
trainingset_test, testingset_test = final_test.randomSplit([0.7, 0.3], seed=2023)

In [0]:
from pyspark.ml.feature import Imputer, VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml import Pipeline, PipelineModel

In [0]:
index = StringIndexer(inputCols=['usual_city'], outputCols=['usual_city_index'])

one_hot = OneHotEncoder(inputCol='usual_city_index', outputCol='usual_city_vector')

final_assemble = VectorAssembler(inputCols=['order_frequency', 'average_delivery_fee', 'average_order_total', 'average_distance_merchant_customer', 'total_discount_vouchers_used', 'average_total_items_in_order', 'average_total_unique_items_in_order', 'total_sessions', 'total_session_duration',"total_subsidy", 'usual_city_vector'], outputCol='input_features')

rft = RandomForestClassifier(featuresCol="input_features", labelCol="test_label", predictionCol="prediction")

In [0]:
# Create a Pipeline and set its stages to include all the preprocessing steps and the classifier.
pipe_test = Pipeline()
pipe_test.setStages(
    [
        index,
        one_hot,
        final_assemble,
        rft
    ]
)

Out[35]: Pipeline_8ad0eaddc16e

In [0]:
# Fit the pipeline to the training data
pipe_model_test = pipe_test.fit(trainingset_test)

In [0]:
# Load the saved pipeline model.
# pipe_model_loaded = PipelineModel.load(chkpt_path)

# Transform the training and test data using the loaded pipeline model.
fitted_train_data = pipe_model_test.transform(trainingset_test)
fitted_test_data = pipe_model_test.transform(testingset_test)

In [0]:
fitted_train_data.groupBy('test_label').pivot('prediction').count().display()

test_label,0.0,1.0
1,5093,518
0,22651,272


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create an evaluator instance with the specified label column, prediction column, and metric name (f1)
evaluator_test = MulticlassClassificationEvaluator(
    labelCol="test_label", predictionCol="prediction", metricName="f1")


In [0]:
# Evaluate the f1-score of the model on the fitted training data
train_metric = evaluator_test.evaluate(fitted_train_data)
print("Training F1-Score = %g" % train_metric)

# Evaluate the f1-score of the model on the fitted test data
test_metric = evaluator_test.evaluate(fitted_test_data)
print("Test F1-Score = %g" % test_metric)

Training F1-Score = 0.750118
Test F1-Score = 0.744899


In [0]:
# Get the input column names used during training
input_columns = final_assemble.getInputCols()

# Get the feature importances from the random forest model
feature_importances = pipe_model_test.stages[-1].featureImportances

# Create a dictionary to map feature indices to feature names
feature_mapping = {index: feature for index, feature in enumerate(input_columns)}

# Print the feature importances with their corresponding feature names
for index, importance in enumerate(feature_importances):
    feature_name = feature_mapping.get(index)
    print(f"Feature {feature_name}: Importance {importance}")


Feature order_frequency: Importance 0.5449491299094827
Feature average_delivery_fee: Importance 0.04442364378898821
Feature average_order_total: Importance 0.0017298792442652269
Feature average_distance_merchant_customer: Importance 0.004556194316766107
Feature total_discount_vouchers_used: Importance 0.011799549202158663
Feature average_total_items_in_order: Importance 0.045266623299046786
Feature average_total_unique_items_in_order: Importance 0.055007488732362435
Feature total_sessions: Importance 0.15951757517007967
Feature total_session_duration: Importance 0.1152935497476822
Feature total_subsidy: Importance 0.01698177056310875
Feature usual_city_vector: Importance 3.699360724441038e-05
Feature None: Importance 7.668183832850624e-05
Feature None: Importance 0.0003293219101616664
Feature None: Importance 3.159867032467583e-05


####Control Pipeline

In [0]:
final_control = spark.read.parquet("dbfs:/FileStore/bdm_data_train/control/final_control.parquet")

In [0]:
trainingset_control, testingset_control = final_control.randomSplit([0.7, 0.3], seed=2023)

In [0]:
index = StringIndexer(inputCols=['usual_city'], outputCols=['usual_city_index'])


one_hot = OneHotEncoder(inputCol='usual_city_index', outputCol='usual_city_vector')

final_assemble = VectorAssembler(inputCols=['order_frequency', 'average_delivery_fee', 'average_order_total', 'average_distance_merchant_customer', 'total_discount_vouchers_used', 'average_total_items_in_order', 'average_total_unique_items_in_order', 'total_sessions', 'total_session_duration',"total_subsidy", 'usual_city_vector'], outputCol='input_features')

rfc = RandomForestClassifier(featuresCol="input_features", labelCol="control_label", predictionCol="prediction")

In [0]:
pipe_control = Pipeline()
pipe_control.setStages(
    [
        index,
        one_hot,
        final_assemble,
        rfc
    ]
)

Out[45]: Pipeline_df7b3dc84f19

In [0]:
pipe_model_control = pipe_control.fit(trainingset_control)

In [0]:
# 12. Load the saved pipeline model.
#pipe_model_loaded = PipelineModel.load(chkpt_path)

#13. Transform the training and test data using the loaded pipeline model.
fitted_train_data = pipe_model_control.transform(trainingset_control)
fitted_test_data = pipe_model_control.transform(testingset_control)

In [0]:
fitted_train_data.groupBy('control_label').pivot('prediction').count().display()

control_label,0.0,1.0
1,610,286
0,4488,134


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Create an evaluator instance with the specified label column, prediction column, and metric name (f1)
evaluator_control = MulticlassClassificationEvaluator(
    labelCol="control_label", predictionCol="prediction", metricName="f1")

In [0]:
# Evaluate the f1-score of the model on the fitted training data
train_metric = evaluator_control.evaluate(fitted_train_data)
print("Training F1-Score = %g" % train_metric)

# Evaluate the f1-score of the model on the fitted test data
test_metric = evaluator_control.evaluate(fitted_test_data)
print("Test F1-Score = %g" % test_metric)

Training F1-Score = 0.844086
Test F1-Score = 0.817247


In [0]:
# Get the input column names used during training
input_columns = final_assemble.getInputCols()

# Get the feature importances from the random forest model
feature_importances = pipe_model_control.stages[-1].featureImportances

# Create a dictionary to map feature indices to feature names
feature_mapping = {index: feature for index, feature in enumerate(input_columns)}

# Print the feature importances with their corresponding feature names
for index, importance in enumerate(feature_importances):
    feature_name = feature_mapping.get(index)
    print(f"Feature {feature_name}: Importance {importance}")


Feature order_frequency: Importance 0.5108730328556326
Feature average_delivery_fee: Importance 0.02424490964170528
Feature average_order_total: Importance 0.011632972032614803
Feature average_distance_merchant_customer: Importance 0.010170019164362255
Feature total_discount_vouchers_used: Importance 0.0587876227938642
Feature average_total_items_in_order: Importance 0.013174691009746265
Feature average_total_unique_items_in_order: Importance 0.017970150088302324
Feature total_sessions: Importance 0.185591870816942
Feature total_session_duration: Importance 0.11583609818088833
Feature total_subsidy: Importance 0.04612698520628512
Feature usual_city_vector: Importance 0.0011704550977126899
Feature None: Importance 0.0008410865706977575
Feature None: Importance 0.0031617876528345994
Feature None: Importance 0.00041831888841175333


> **Why did you choose each step of your ML Pipeline?**

- 1. Create a StringIndexer to convert the 'subsidy_bucket' column into a numerical index column called 'subsidy_bucket_idx'.

- 2. Create a OneHotEncoder to convert the 'usual_city' column into a one-hot encoded vector column called 'usual_city_vector'.

- 3. Create a VectorAssembler to combine the features and the one-hot encoded categorical feature into a single vector column called 'input_features'.

- 4. Create a RandomForestClassifier with the specified input and output columns.

Note1: We did not really need any kind of preprocessing step because our data was previously treated so the dataframes passed were clean.

Note2: We did not scale our features because the model used was a randomforest, scaling is not usually made when using decision trees. Vectorizing was also not done because we needed to check featureImportance.

#### Step 3: Tuning the hyperparameters of your model

In [0]:
experiment_name = "/Shared/test_cv_tuner"
mlflow.set_experiment(experiment_name)

Out[57]: <Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2533558749698245', creation_time=1686421297594, experiment_id='2533558749698245', last_update_time=1686428613434, lifecycle_stage='active', name='/Shared/test_cv_tuner', tags={'mlflow.experiment.sourceName': '/Shared/test_cv_tuner',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'daniel.branco@live.com.pt',
 'mlflow.ownerId': '1963320249234256'}>

In [0]:
from functools import reduce
import numpy as np
import pandas as pd

# Define the parameter grid for maxDepth and numBins
num_trees = [5,10,20]
max_depths = [2, 4, 8]
num_bins = [2, 4, 8]

# Create a pandas DataFrame with the parameter combinations
param_grid = pd.DataFrame([(tree, depth, bins) for tree in num_trees for depth in max_depths for bins in num_bins],
                          columns=['num_trees', 'max_depth', 'num_bins'])

# Perform manual 3-fold cross-validation
num_folds = 3
probs = [0.33, 0.33, 0.33] 
train_data_to_split_test = trainingset_test.cache()
splits = train_data_to_split_test.randomSplit(probs, seed=42)

In [0]:
evaluator_test = MulticlassClassificationEvaluator(
    labelCol="test_label", predictionCol="prediction", metricName="f1")

In [0]:
with mlflow.start_run():
    active_run = mlflow.active_run()
    expId = active_run.info.experiment_id
    print(f"Exp ID: {expId}")
    
    for _, row in param_grid.iterrows():
        trees = row['num_trees']
        depth = row['max_depth']
        bins = row['num_bins']
        
        # Set the parameters for the RandomForestClassifier
        pipe_test.getStages()[-1].setParams(numTrees=trees,maxDepth=depth, maxBins=bins)

        # Initialize the metrics for x-val
        metrics = {
          'accuracy': [],
          'precision': [],
          'recall': [],
          'f1': []
        }

        # Log the parameters and evaluation metrics using MLflow
        with mlflow.start_run(nested=True): 
            
            # Perform k-fold cross-validation
            for i in range(num_folds):
                # Split the data into training and test sets
                validation_df = splits[i]
                train_splits = splits.copy()
                train_splits.pop(i)
                train_df = reduce(lambda x,y: x.union(y), train_splits)

                # Train the model
                model = pipe_test.fit(train_df)

                # Make predictions on the test set
                predictions = model.transform(validation_df)

                
                # Evaluate the model
                accuracy = evaluator_test.evaluate(predictions,
                                {evaluator_test.metricName: "accuracy"})
                precision = evaluator_test.evaluate(predictions, 
                        {evaluator_test.metricName: "weightedPrecision"})
                recall = evaluator_test.evaluate(predictions,
                           {evaluator_test.metricName: "weightedRecall"})
                f1 = evaluator_test.evaluate(predictions,
                                       {evaluator_test.metricName: "f1"})
               
                # Append the metrics to the dictionary
                metrics['accuracy'].append(accuracy)
                metrics['precision'].append(precision)
                metrics['recall'].append(recall)
                metrics['f1'].append(f1)
            
            # Calculate the average metrics across the folds
            avg_accuracy = np.mean(metrics['accuracy'])
            avg_precision = np.mean(metrics['precision'])
            avg_recall = np.mean(metrics['recall'])
            avg_f1 = np.mean(metrics['f1'])
        
            mlflow.log_param("num_trees", trees)
            mlflow.log_param("max_depth", depth)
            mlflow.log_param("num_bins", bins)
            mlflow.log_metric("avg_accuracy", avg_accuracy)
            mlflow.log_metric("avg_precision", avg_precision)
            mlflow.log_metric("avg_recall", avg_recall)
            mlflow.log_metric("avg_f1", avg_f1)

            print(f"NumTrees: {trees}, MaxDepth: {depth}, NumBins:{bins}")
            print(f"Avg Accuracy: {avg_accuracy}")
            print(f"Avg Precision: {avg_precision}")
            print(f"Avg Recall: {avg_recall}")
            print(f"Avg F1: {avg_f1}")
            mlflow.spark.log_model(model, "model", pip_requirements="requirements.txt") # passing the requirements speed up the log_model process

train_data_to_split_test.unpersist()

Exp ID: 2533558749698245
NumTrees: 5, MaxDepth: 2, NumBins:2
Avg Accuracy: 0.8033372862998333
Avg Precision: 0.6453607149701936
Avg Recall: 0.8033372862998333
Avg F1: 0.7157328266204092


/databricks/python/lib/python3.9/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


NumTrees: 5, MaxDepth: 2, NumBins:4
Avg Accuracy: 0.8033372862998333
Avg Precision: 0.6453607149701936
Avg Recall: 0.8033372862998333
Avg F1: 0.7157328266204092
NumTrees: 5, MaxDepth: 2, NumBins:8
Avg Accuracy: 0.8033372862998333
Avg Precision: 0.6453607149701936
Avg Recall: 0.8033372862998333
Avg F1: 0.7157328266204092
NumTrees: 5, MaxDepth: 4, NumBins:2
Avg Accuracy: 0.8033372862998333
Avg Precision: 0.6453607149701936
Avg Recall: 0.8033372862998333
Avg F1: 0.7157328266204092
NumTrees: 5, MaxDepth: 4, NumBins:4
Avg Accuracy: 0.8033372862998333
Avg Precision: 0.6453607149701936
Avg Recall: 0.8033372862998333
Avg F1: 0.7157328266204092
NumTrees: 5, MaxDepth: 4, NumBins:8
Avg Accuracy: 0.8034064928467727
Avg Precision: 0.7097332823643692
Avg Recall: 0.8034064928467727
Avg F1: 0.7158987010828373
NumTrees: 5, MaxDepth: 8, NumBins:2
Avg Accuracy: 0.8031636854794337
Avg Precision: 0.6914736702159546
Avg Recall: 0.8031636854794337
Avg F1: 0.7159769966966479
NumTrees: 5, MaxDepth: 8, NumBins:

In [0]:
#Read data from an MLflow experiment using Apache Spark. The spark object represents the SparkSession, and expId is the ID of the MLflow experiment we want to load. The data is loaded into a DataFrame called metrics_df.
metrics_df = spark.createDataFrame(mlflow.search_runs())

display(metrics_df)

run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.avg_recall,metrics.avg_accuracy,metrics.avg_precision,metrics.avg_f1,params.num_bins,params.num_trees,params.max_depth,tags.mlflow.databricks.notebookRevisionID,tags.mlflow.databricks.workspaceID,tags.mlflow.source.name,tags.mlflow.databricks.notebookPath,tags.mlflow.databricks.notebookID,tags.mlflow.parentRunId,tags.mlflow.log-model.history,tags.mlflow.databricks.notebook.commandID,tags.mlflow.rootRunId,tags.mlflow.source.type,tags.mlflow.databricks.webappURL,tags.mlflow.databricks.cluster.libraries,tags.mlflow.user,tags.mlflow.databricks.workspaceURL,tags.mlflow.runName,tags.mlflow.databricks.cluster.info,tags.mlflow.databricks.cluster.id
e51fc51fb4e548d399ab136ca11fd028,2533558749698245,FINISHED,dbfs:/databricks/mlflow-tracking/2533558749698245/e51fc51fb4e548d399ab136ca11fd028/artifacts,2023-06-11T18:29:39.254+0000,2023-06-11T18:31:12.705+0000,0.8095777872856935,0.8095777872856935,0.7725066947234606,0.7674913724603546,8,20,8,1686508272871,6018455593152283,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,3640989080997027,c58b7c72cd3f4edb8d4a07c2ab21759f,"[{""artifact_path"":""model"",""flavors"":{""spark"":{""pyspark_version"":""3.3.2.dev0"",""model_data"":""sparkml"",""code"":null},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.9.5"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""run_id"":""e51fc51fb4e548d399ab136ca11fd028"",""model_uuid"":""3bb39287d5e5486697869ad446a6511b"",""utc_time_created"":""2023-06-11 18:31:12.080080"",""mlflow_version"":""2.4.1"",""databricks_runtime"":""12.2.x-scala2.12""}]",6044476463787687796_7085695313648202206_d733137b853d49d297739aab3383f49c,c58b7c72cd3f4edb8d4a07c2ab21759f,NOTEBOOK,https://community.cloud.databricks.com,"{""installable"":[],""redacted"":[]}",daniel.branco@live.com.pt,https://community.cloud.databricks.com,skittish-newt-492,"{""cluster_name"":""My Cluster"",""spark_version"":""12.2.x-scala2.12"",""node_type_id"":""dev-tier-node"",""driver_node_type_id"":""dev-tier-node"",""autotermination_minutes"":120,""disk_spec"":{""disk_count"":0},""num_workers"":0}",0611-092241-srtkwcu9
c40d61ae4b2c48f89522d36c462365ac,2533558749698245,FINISHED,dbfs:/databricks/mlflow-tracking/2533558749698245/c40d61ae4b2c48f89522d36c462365ac/artifacts,2023-06-11T18:28:13.268+0000,2023-06-11T18:29:39.145+0000,0.8034461814909996,0.8034461814909996,0.7523880231578869,0.7329120159686795,4,20,8,1686508179258,6018455593152283,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,3640989080997027,c58b7c72cd3f4edb8d4a07c2ab21759f,"[{""artifact_path"":""model"",""flavors"":{""spark"":{""pyspark_version"":""3.3.2.dev0"",""model_data"":""sparkml"",""code"":null},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.9.5"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""run_id"":""c40d61ae4b2c48f89522d36c462365ac"",""model_uuid"":""849780b46ee9428c89a79f5097a2e310"",""utc_time_created"":""2023-06-11 18:29:38.552557"",""mlflow_version"":""2.4.1"",""databricks_runtime"":""12.2.x-scala2.12""}]",6044476463787687796_7085695313648202206_d733137b853d49d297739aab3383f49c,c58b7c72cd3f4edb8d4a07c2ab21759f,NOTEBOOK,https://community.cloud.databricks.com,"{""installable"":[],""redacted"":[]}",daniel.branco@live.com.pt,https://community.cloud.databricks.com,overjoyed-flea-20,"{""cluster_name"":""My Cluster"",""spark_version"":""12.2.x-scala2.12"",""node_type_id"":""dev-tier-node"",""driver_node_type_id"":""dev-tier-node"",""autotermination_minutes"":120,""disk_spec"":{""disk_count"":0},""num_workers"":0}",0611-092241-srtkwcu9
2ab020be4bab47478006890999c6ab22,2533558749698245,FINI

In [0]:
experiment_name = "/Shared/control_cv_tuner"
mlflow.set_experiment(experiment_name)

Out[70]: <Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2533558749698254', creation_time=1686423507353, experiment_id='2533558749698254', last_update_time=1686429929904, lifecycle_stage='active', name='/Shared/control_cv_tuner', tags={'mlflow.experiment.sourceName': '/Shared/control_cv_tuner',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'daniel.branco@live.com.pt',
 'mlflow.ownerId': '1963320249234256'}>

In [0]:
evaluator_control = MulticlassClassificationEvaluator(
    labelCol="control_label", predictionCol="prediction", metricName="f1")

In [0]:
from functools import reduce
import numpy as np
import pandas as pd

# Define the parameter grid for maxDepth and numBins
num_trees = [5,10,20]
max_depths = [2, 4, 8]
num_bins = [2, 4, 8]

# Create a pandas DataFrame with the parameter combinations
param_grid = pd.DataFrame([(tree, depth, bins) for tree in num_trees for depth in max_depths for bins in num_bins],
                          columns=['num_trees', 'max_depth', 'num_bins'])

# Perform manual 3-fold cross-validation
num_folds = 3
probs = [0.33, 0.33, 0.33] 
train_data_to_split_control = trainingset_control.cache()
splits = train_data_to_split_control.randomSplit(probs, seed=42)

In [0]:
with mlflow.start_run():
    active_run = mlflow.active_run()
    expId = active_run.info.experiment_id
    print(f"Exp ID: {expId}")
    
    for _, row in param_grid.iterrows():
        trees = row['num_trees']
        depth = row['max_depth']
        bins = row['num_bins']
        
        # Set the parameters for the RandomForestClassifier
        pipe_control.getStages()[-1].setParams(numTrees = trees, maxDepth=depth, maxBins=bins)

        # Initialize the metrics for x-val
        metrics = {
          'accuracy': [],
          'precision': [],
          'recall': [],
          'f1': []
        }

        # Log the parameters and evaluation metrics using MLflow
        with mlflow.start_run(nested=True): 
            
            # Perform k-fold cross-validation
            for i in range(num_folds):
                # Split the data into training and test sets
                validation_df = splits[i]
                train_splits = splits.copy()
                train_splits.pop(i)
                train_df = reduce(lambda x,y: x.union(y), train_splits)

                # Train the model
                model = pipe_control.fit(train_df)

                # Make predictions on the test set
                predictions = model.transform(validation_df)

                # Evaluate the model
                accuracy = evaluator_control.evaluate(predictions,
                                {evaluator_control.metricName: "accuracy"})
                precision = evaluator_control.evaluate(predictions, 
                        {evaluator_control.metricName: "weightedPrecision"})
                recall = evaluator_control.evaluate(predictions,
                           {evaluator_control.metricName: "weightedRecall"})
                f1 = evaluator_control.evaluate(predictions,
                                       {evaluator_control.metricName: "f1"})
               
                # Append the metrics to the dictionary
                metrics['accuracy'].append(accuracy)
                metrics['precision'].append(precision)
                metrics['recall'].append(recall)
                metrics['f1'].append(f1)
            
            # Calculate the average metrics across the folds
            avg_accuracy = np.mean(metrics['accuracy'])
            avg_precision = np.mean(metrics['precision'])
            avg_recall = np.mean(metrics['recall'])
            avg_f1 = np.mean(metrics['f1'])

            print(f"NumTrees: {trees}, MaxDepth: {depth}, NumBins:{bins}")
            print(f"Avg Accuracy: {avg_accuracy}")
            print(f"Avg Precision: {avg_precision}")
            print(f"Avg Recall: {avg_recall}")
            print(f"Avg F1: {avg_f1}")
            mlflow.spark.log_model(model, "model", pip_requirements="requirements.txt") # passing the requirements speed up the log_model process

train_data_to_split_control.unpersist()

Exp ID: 2533558749698254
NumTrees: 5, MaxDepth: 2, NumBins:2
Avg Accuracy: 0.8375775299531849
Avg Precision: 0.7015807294143249
Avg Recall: 0.8375775299531849
Avg F1: 0.7635588830459573
NumTrees: 5, MaxDepth: 2, NumBins:4
Avg Accuracy: 0.8375775299531849
Avg Precision: 0.7015807294143249
Avg Recall: 0.8375775299531849
Avg F1: 0.7635588830459573
NumTrees: 5, MaxDepth: 2, NumBins:8
Avg Accuracy: 0.8375775299531849
Avg Precision: 0.7015807294143249
Avg Recall: 0.8375775299531849
Avg F1: 0.7635588830459573
NumTrees: 5, MaxDepth: 4, NumBins:2
Avg Accuracy: 0.8375775299531849
Avg Precision: 0.7015807294143249
Avg Recall: 0.8375775299531849
Avg F1: 0.7635588830459573
NumTrees: 5, MaxDepth: 4, NumBins:4
Avg Accuracy: 0.83943627789035
Avg Precision: 0.7656350428735434
Avg Recall: 0.83943627789035
Avg F1: 0.778108149994052
NumTrees: 5, MaxDepth: 4, NumBins:8
Avg Accuracy: 0.8412215561748176
Avg Precision: 0.8061359469309434
Avg Recall: 0.8412215561748176
Avg F1: 0.7821978701715168
NumTrees: 5, M

In [0]:
metrics_df = spark.createDataFrame(mlflow.search_runs())

display(metrics_df)

run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.avg_accuracy,params.num_bins,params.num_trees,params.max_depth,tags.mlflow.databricks.notebookRevisionID,tags.mlflow.databricks.workspaceID,tags.mlflow.source.name,tags.mlflow.databricks.notebookPath,tags.mlflow.databricks.notebookID,tags.mlflow.parentRunId,tags.mlflow.log-model.history,tags.mlflow.databricks.notebook.commandID,tags.mlflow.rootRunId,tags.mlflow.source.type,tags.mlflow.databricks.webappURL,tags.mlflow.databricks.cluster.libraries,tags.mlflow.user,tags.mlflow.databricks.workspaceURL,tags.mlflow.runName,tags.mlflow.databricks.cluster.info,tags.mlflow.databricks.cluster.id
4b0cf0e083e447129d44bdfb46fac7fa,2533558749698254,FINISHED,dbfs:/databricks/mlflow-tracking/2533558749698254/4b0cf0e083e447129d44bdfb46fac7fa/artifacts,2023-06-11T18:51:17.178+0000,2023-06-11T18:52:33.459+0000,null,null,null,null,1686509553634,6018455593152283,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,3640989080997027,5b0d1c94d78e4fb8bd7a20d1c15bdab9,"[{""artifact_path"":""model"",""flavors"":{""spark"":{""pyspark_version"":""3.3.2.dev0"",""model_data"":""sparkml"",""code"":null},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.9.5"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""run_id"":""4b0cf0e083e447129d44bdfb46fac7fa"",""model_uuid"":""0127e71eba72499c9ec709e3ef495107"",""utc_time_created"":""2023-06-11 18:52:32.799272"",""mlflow_version"":""2.4.1"",""databricks_runtime"":""12.2.x-scala2.12""}]",6044476463787687796_4741342959409093151_e30fd4176ee7450589c8e3c9272d250d,5b0d1c94d78e4fb8bd7a20d1c15bdab9,NOTEBOOK,https://community.cloud.databricks.com,"{""installable"":[],""redacted"":[]}",daniel.branco@live.com.pt,https://community.cloud.databricks.com,ambitious-mule-365,"{""cluster_name"":""My Cluster"",""spark_version"":""12.2.x-scala2.12"",""node_type_id"":""dev-tier-node"",""driver_node_type_id"":""dev-tier-node"",""autotermination_minutes"":120,""disk_spec"":{""disk_count"":0},""num_workers"":0}",0611-092241-srtkwcu9
32cd69bb60034040be32fcd313b59a0d,2533558749698254,FINISHED,dbfs:/databricks/mlflow-tracking/2533558749698254/32cd69bb60034040be32fcd313b59a0d/artifacts,2023-06-11T18:50:02.308+0000,2023-06-11T18:51:17.041+0000,null,null,null,null,1686509477231,6018455593152283,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,/Users/daniel.branco@live.com.pt/Project/Final Project Blueprint - 20220599 20220646,3640989080997027,5b0d1c94d78e4fb8bd7a20d1c15bdab9,"[{""artifact_path"":""model"",""flavors"":{""spark"":{""pyspark_version"":""3.3.2.dev0"",""model_data"":""sparkml"",""code"":null},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.9.5"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""run_id"":""32cd69bb60034040be32fcd313b59a0d"",""model_uuid"":""b9db305a1ff14148b219ffa22906a51e"",""utc_time_created"":""2023-06-11 18:51:16.506210"",""mlflow_version"":""2.4.1"",""databricks_runtime"":""12.2.x-scala2.12""}]",6044476463787687796_4741342959409093151_e30fd4176ee7450589c8e3c9272d250d,5b0d1c94d78e4fb8bd7a20d1c15bdab9,NOTEBOOK,https://community.cloud.databricks.com,"{""installable"":[],""redacted"":[]}",daniel.branco@live.com.pt,https://community.cloud.databricks.com,grandiose-shark-838,"{""cluster_name"":""My Cluster"",""spark_version"":""12.2.x-scala2.12"",""node_type_id"":""dev-tier-node"",""driver_node_type_id"":""dev-tier-node"",""autotermination_minutes"":120,""disk_spec"":{""disk_count"":0},""num_workers"":0}",0611-092241-srtkwcu9
2288fb3539ee4cefa6a98f71a7ef5814,2533558749698254,FINISHED,dbfs:/databricks/mlflow-tracking/2533558749698254/2288fb3539ee4cefa6a98f71a7ef5814/artifacts,2023-06-11T18:48:55.602+0000,2023-06-11T18:50:02.192+0000,null,null,null,null,168

>**Which metrics did you choose for evaluation? Justify.**
- Accuracy: Measures the overall correctness of the model's predictions.
- Precision: Evaluates the model's ability to correctly identify positive instances among all predicted positive instances.
- Recall: Assesses the model's ability to identify all positive instances correctly.
- F1-Score: Provides a balanced measure between precision and recall, useful when both false positives and false negatives are equally important.

>**Present the results of your hyperparameter tuning in the next cell. Is there any parameter more correlated to the evaluation metrics? What is your hypothesis about this?**


The best parameters for the test dataframe
- NumTrees: 20 
- MaxDepth: 8 
- NumBins:8
- Avg Accuracy: 0.8095777872856935
- Avg Precision: 0.7725066947234606
- Avg Recall: 0.8095777872856935
- Avg F1: 0.7674913724603546

The best parameters for the control dataframe
- NumTrees: 20 
- MaxDepth: 8
- NumBins:8
- Avg Accuracy: 0.8549314992300304
- Avg Precision: 0.8340326178228055
- Avg Recall: 0.8549314992300304
- Avg F1: 0.8362696465226471

Basically the best results we got were for the highest parameters we set, the model could have improved further if we experienced with more but due to time constraints we think this was a good enough search space

...

#### Step 4: Generate the targetting with your model and analyze the results of `JUNE`

As we explained in the beggining, we will now use the unsupervised learning to obtain 5 different segments and see if any of them are worth giving a voucher to

In [0]:
from pyspark.ml.linalg import Vectors

# Get the feature importances from the test random forest model
test_feature_importances = Vectors.dense(pipe_model_test.stages[-1].featureImportances)

# Get the feature importances from the control random forest model
control_feature_importances = Vectors.dense(pipe_model_control.stages[-1].featureImportances)

# Calculate the difference in feature importances between test and control models
importance_diff = test_feature_importances - control_feature_importances.toArray()

# Get the indices of features where the importance difference is positive
positive_importance_indices = [i for i, importance in enumerate(importance_diff) if importance > 0]

# Get the group of characteristics that have higher importance for a purchase when a voucher is given
group_of_characteristics = [trainingset_test.columns[i] for i in positive_importance_indices]

# Print the group of characteristics
print("Group of Characteristics with Higher Importance for Purchase when Voucher Given:")
print(positive_importance_indices)



Group of Characteristics with Higher Importance for Purchase when Voucher Given:
[0, 1, 5, 6]


Features a avaliar, Order_frequency, average_delivery_fee, average_total_items_in_order, average_total_unique_items_in_order

In [0]:
orders_k = orders_df.filter((f.col('reference_date') >= '2021-06-01') & (f.col('reference_date') < '2021-06-31'))
sessions_k = sessions_df.filter((f.col('reference_date') >= '2021-06-01') & (f.col('reference_date') < '2021-06-31'))
items_k = items_df.filter((f.col('reference_date') >= '2021-06-01') & (f.col('reference_date') < '2021-06-31'))

In [0]:
items_k = generate_features(items_k, '2021-06-01', '2021-06-31')
orders_k = generate_features(orders_k, '2021-06-01', '2021-06-31')
sessions_k = generate_features(sessions_k, '2021-06-01', '2021-06-31')

In [0]:
# Assuming df1, df2, and df3 are the three DataFrames

# Selecting specific columns from df1
orders_select = orders_k.select("account_id","reference_date","order_id", "order_frequency", "average_delivery_fee")

# Selecting specific columns from df2
items_select = items_k.select("order_id", "total_items_in_order", "total_unique_items_in_order")

# Joining the selected DataFrames
final_df = orders_select.join(items_select, on="order_id", how="inner")


In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

# Assuming you have a dataframe called df with the significant features

# Create a vector assembler to combine the significant features into a single feature column
assembler = VectorAssembler(inputCols=["order_frequency", "average_delivery_fee", "total_items_in_order", "total_unique_items_in_order"], outputCol="features")
df = assembler.transform(final_df)

# Apply K-means clustering to create customer segments
k = 5  # Number of desired segments
kmeans = KMeans(k=k, seed=42)
model = kmeans.fit(df)

# Assign a segment label to each customer based on the clustering result
df = model.transform(df)
df = df.withColumn("segment", df["prediction"])

# Analyze the characteristics and behavior of each customer segment
segment_counts = df.groupBy("segment").count().orderBy("segment")
segment_counts.show()

# Use the customer segmentation for personalized marketing or targeted campaigns
# Example: Send different offers or recommendations based on the segment
df.select("account_id", "segment").show()

+-------+------+
|segment| count|
+-------+------+
|      0| 63709|
|      1|164141|
|      2|187576|
|      3|158162|
|      4|125662|
+-------+------+

+--------------------+-------+
|          account_id|segment|
+--------------------+-------+
|20466b8aa08e5b191...|      1|
|c51bc99ac4304a93c...|      1|
|52eea6f23bd0ef59a...|      1|
|40e09c22fadd0ffae...|      1|
|21e6285eb6b690134...|      3|
|3b3d5174a131c06a3...|      3|
|63e33d4a1d99a223a...|      2|
|2620617631824b7e0...|      3|
|53338bde0e6bcb595...|      3|
|485781649e96208ea...|      3|
|157fa050a46b2feed...|      1|
|0e3d46c479179f940...|      2|
|0fa48ebf08d21664a...|      3|
|4dd22e01543a60373...|      1|
|6859cfde4d80f23d7...|      1|
|4f020ca266e5314fd...|      0|
|6241281f7af9011b5...|      0|
|587002fdd871f9b73...|      1|
|11177e17a277c9074...|      2|
|587e3ff8f4ec4c697...|      3|
+--------------------+-------+
only showing top 20 rows



In [0]:
from pyspark.sql import functions as f
# Group the data by segment and compute summary statistics for each segment
segment_characteristics = df.groupby("segment").agg(
    f.avg("order_frequency").alias("avg_order_frequency"),
    f.avg("average_delivery_fee").alias("avg_delivery_fee"),
    f.avg("total_items_in_order").alias("avg_total_items_in_order"),
    f.avg("total_unique_items_in_order").alias("avg_total_unique_items_in_order")
)

# Show the characteristics of each segment
segment_characteristics.show()


+-------+-------------------+------------------+------------------------+-------------------------------+
|segment|avg_order_frequency|  avg_delivery_fee|avg_total_items_in_order|avg_total_unique_items_in_order|
+-------+-------------------+------------------+------------------------+-------------------------------+
|      1| 18.036139660413912|234.03877766273132|      3.1719619107962056|              2.335266630518883|
|      3| 17.503559641380356| 92.23083010511304|      2.6521288299338655|             2.0140868223719983|
|      4| 16.573626076299917|  25.2562806884834|       2.201580430042495|               1.72661584249813|
|      2| 18.003177378769138|159.19518200107706|       2.985717789056169|             2.2123832473237512|
|      0| 15.949222244894756|339.83199155535146|      3.1582507965907487|             2.3553501075201306|
+-------+-------------------+------------------+------------------------+-------------------------------+



In [0]:
orders_sample = spark.read.parquet(f"/FileStore/bdm_data_train/train/ord_sample")
test_sample = spark.read.parquet(f"/FileStore/bdm_data_train/train/mkt_test_assignment_sample")

In [0]:
orders_segment = orders_sample.join(df.select("account_id","segment"), on = "account_id", how = "inner")

In [0]:
def test_results_applying_target(test_month: str = '2021-06-01', target: DataFrame = None):

    orders_sample = spark.read.parquet(f"/FileStore/bdm_data_train/train/ord_sample")
    test_sample = spark.read.parquet(f"/FileStore/bdm_data_train/train/mkt_test_assignment_sample")
  
    orders_by_account = (
      orders_sample
      .filter( f.date_trunc('month', f.col('reference_date'))=='2021-06-01')
      .groupBy('account_id', 'reference_date')
      .agg(
          f.countDistinct('order_id').alias('orders'),
          f.sum('discount_vouchers_used').alias('vouchers_used'),
          f.sum('subsidy').alias('subsidy'),
          f.avg('order_total').alias('aov')
      )
    )

    if target:
      test_sample = ( test_sample.join(target, ['account_id'], 'inner') )

    (
      test_sample
      .filter(f.col('reference_month')==test_month)
      .join(orders_by_account,  ['account_id', 'reference_date'], 'left')
      .groupBy('test_assignment')
      .agg(
          f.count('account_id').alias('test_size'),
          f.round((f.sum( f.when(f.col('orders') > 0, 1).otherwise(0) )), 3).alias('conversions'),
          f.round((f.sum( f.when(f.col('orders') > 0, 1).otherwise(0) ) / f.count('account_id') ), 3).alias('conversion_rate'),
          f.round(f.sum('subsidy'), 0).alias('sub_total'),
          (f.sum( f.when(f.col('vouchers_used') > 0, 1).otherwise(0) ) ).alias('vouchers_used'),
          f.avg('aov').alias('aov')
      )
      .display()
      
    )

In [0]:
target_customers =( 
  orders_segment
  .filter(f.col('reference_date')>='2021-01-01')
  .filter(f.col('reference_date')<'2021-06-01')
  .filter(f.col('segment')==0)
  .groupBy('account_id')
  .agg(
      f.countDistinct('segment').alias('segment')
  )
  .select('account_id')
  .distinct()
)

In [0]:
test_results_applying_target('2021-06-01', target = target_customers)

test_assignment,test_size,conversions,conversion_rate,sub_total,vouchers_used,aov
control,4059,973,0.24,0.0,0,3620.6239208633133
test,4533,1183,0.261,17577.0,46,3552.1653106508925


In [0]:
IRR = 0.261 - 0.24 # Conversion(test) - Conversion(control)
IRR

Out[139]: 0.02100000000000002

In [0]:
MUBi = IRR * 4533 # conversions # IRR * N(test)
MUBi

Out[142]: 95.19300000000008

In [0]:
Cannib = (1183-MUBi)/1183 # [Conversions with subsidy - (IRR x N(test)]/Conversions with subsidy
Cannib

Out[145]: 0.9195325443786982

In [0]:
NIR = ((3552 * 0.261) - (3620 * 0.24)) * 4533# = $MU 130,182 # Revenue(test) - Revenue(control)
NIR

Out[149]: 264146.9760000002

In [0]:
CPI = 17577/46 #= MU$ 912 # Investment(test)/MUBi
CPI

Out[150]: 382.10869565217394

In [0]:
PL = NIR - 17577
PL

Out[153]: 246569.9760000002

> **Analyze the results. What are your conclusions? Remember to compare the results to the baseline targetting.**

From the different segments we got, segment 0 seemed to have the better results, so it was the one we went with.

Compared to the baseline, we can say that our test was slightly better, with an:
- IRR of 2.1% vs an IRR of 0%
- Incremental MUB of 95 vs 0
- Cannibalization rate of 92% vs 100%
- Cost per increment of 382 vs N/A
- And the resulting P/L of 246569 vs -330

Overall, we managed to get a slightly better result but we would say that this is still not a good target and further studies and approaches should be evaluated to get a better target segment.
Sending the promotional campaign to everyone resulted in ZERO incremental conversions and a resulting loss of - MU$ 330 K. If we sent this promotional campaign only to users with an Average Order Values below MU $ in the previous month, we'd have an incremental response rate of 2.1%, at MU$382 for each incremental conversion, resulting in a PROFIT of MU$246K.

#### Step 4: Generate the targetting with your model and analyze the results of `JULY`. 
*(the data of july will be made available 24 hours before the submission deadline)*

In [0]:
orders_df_july = spark.read.parquet(f"/FileStore/bdm_data_test/ord_sample")
ab_test_df_july = spark.read.parquet(f"/FileStore/bdm_data_test/test/mkt_test_assignment_sample")

In [0]:
orders_segment_july = orders_df_july.join(df.select("account_id","segment"), on = "account_id", how = "inner")
#joined_df_july = ab_test_df_july.join(orders_df_july, on=['account_id','reference_date'], how='left')

In [0]:
def test_results_applying_target(test_month: str = '2021-07-01', target: DataFrame = None):

    orders_sample = spark.read.parquet(f"/FileStore/bdm_data_test/ord_sample")
    test_sample = spark.read.parquet(f"/FileStore/bdm_data_test/test/mkt_test_assignment_sample")

    orders_by_account = (
      orders_sample
      .filter( f.date_trunc('month', f.col('reference_date'))=='2021-07-01')
      .groupBy('account_id', 'reference_date')
      .agg(
          f.countDistinct('order_id').alias('orders'),
          f.sum('discount_vouchers_used').alias('vouchers_used'),
          f.sum('subsidy').alias('subsidy'),
          f.avg('order_total').alias('aov')
      )
    )

    if target:
      test_sample = ( test_sample.join(target, ['account_id'], 'inner') )

    (
      test_sample
      .filter(f.col('reference_month')==test_month)
      .join(orders_by_account,  ['account_id', 'reference_date'], 'left')
      .groupBy('test_assignment')
      .agg(
          f.count('account_id').alias('test_size'),
          f.round((f.sum( f.when(f.col('orders') > 0, 1).otherwise(0) )), 3).alias('conversions'),
          f.round((f.sum( f.when(f.col('orders') > 0, 1).otherwise(0) ) / f.count('account_id') ), 3).alias('conversion_rate'),
          f.round(f.sum('subsidy'), 0).alias('sub_total'),
          (f.sum( f.when(f.col('vouchers_used') > 0, 1).otherwise(0) ) ).alias('vouchers_used'),
          f.avg('aov').alias('aov')
      )
      .display()
      
    )

In [0]:
target_customers =( 
  orders_segment
  .filter(f.col('reference_date')>='2021-02-01')
  .filter(f.col('reference_date')<'2021-07-01')
  .filter(f.col('segment')==4)
  .groupBy('account_id')
  .agg(
      f.countDistinct('segment').alias('segment')
  )
  .select('account_id')
  .distinct()
)

In [0]:
test_results_applying_target('2021-07-01', target = target_customers)

test_assignment,test_size,conversions,conversion_rate,sub_total,vouchers_used,aov
control,9254,1718,0.186,0.0,0,1661.130029346139
test,10791,2325,0.215,134743.0,293,1505.0521236559148


In [0]:
IRR = 0.215 - 0.186 # Conversion(test) - Conversion(control)
IRR

Out[156]: 0.028999999999999998

In [0]:
MUBi = IRR * 10791 # conversions # IRR * N(test)
MUBi

Out[157]: 312.93899999999996

In [0]:
Cannib = (2325-MUBi)/2325 # [Conversions with subsidy - (IRR x N(test)]/Conversions with subsidy
Cannib

Out[158]: 0.8654025806451614

In [0]:
NIR = ((1505 * 0.215) - (1661 * 0.186)) * 10791# = $MU 130,182 # Revenue(test) - Revenue(control)
NIR

Out[159]: 157861.5390000002

In [0]:
CPI = 134743/293 #= MU$ 912 # Investment(test)/MUBi
CPI

Out[160]: 459.8737201365188

In [0]:
PL = NIR - 134743
PL

Out[161]: 23118.539000000193

> **Results Analysis**

From the different segments we got, segment 4 seemed to have the better results, so it was the one we went with.

The results had were:
- IRR of 2.9%
- Incremental MUB of 313
- Cannibalization rate of 87%
- Cost per increment of 460
- And the resulting P/L of 23119

In conclusion, we were actually a bit surprised by segment 4 performance since it didnt perform nearly as good as in june. But we still consider this results poor and would reccomend using a different metric to find the best segment. Although the results are not very good in our opinion the company is still making money although the results are not very good in our opinion the company is still making money with the chosen segmentwith the chosen segment.